# 3. Optimism-Guided Data Augmentation
Implements Algorithm 2 from the paper: Explore high Q-value actions to augment dataset.

In [ ]:
import pickle
import numpy as np
import d3rlpy
from d3rlpy.dataset import MDPDataset

# Load dataset
with open('offline_dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

states = np.array([d['state'] for d in dataset], dtype=np.float32)
actions = np.array([d['action'] for d in dataset], dtype=np.int32)
rewards = np.array([d['reward'] for d in dataset], dtype=np.float32)
terminals = np.array([float(d['done']) for d in dataset], dtype=np.float32)

mdp_ds = MDPDataset(observations=states, actions=actions, rewards=rewards, terminals=terminals, discrete_action=True)

# Build and Load model
cql = d3rlpy.algos.DiscreteCQLConfig().create()
cql.build_with_dataset(mdp_ds)
cql.load_model("cql_tutor.pt")

ACTION_SPACE_SIZE = 4


In [ ]:
# Find Optimistic Actions
optimistic_states = []

for idx, d in enumerate(dataset):
    s = d['state']
    a_baseline = d['action']
    
    # Q-values for all actions on this state
    action_batch = np.array([0, 1, 2, 3])
    state_batch = np.array([s for _ in range(4)])
    
    q_values = cql.predict_value(state_batch, action_batch)
    a_opt = np.argmax(q_values)
    
    # If a better action is found and it's not what was chosen in the dataset
    if a_opt != a_baseline and q_values[a_opt] > q_values[a_baseline]:
        optimistic_states.append({
            'state_idx': idx,
            'optimistic_action': a_opt,
            'q_opt': q_values[a_opt],
            'q_base': q_values[a_baseline]
        })

print(f"Found {len(optimistic_states)} optimistic action opportunities.")
